In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# then paste the whole limitation_v_vi.py contents in the next cell and run

Mounted at /content/drive


In [ ]:
#!/usr/bin/env python3
"""
Resolve paper limitations (v) and (vi) from the existing evaluation data.

(v)  Multi-turn vs single-turn: a DIRECT effect-size test of hallucination
     rates (replaces the indirect detectability-ablation argument).
(vi) Binary outcome modeled with LOGISTIC regression (model x technique),
     replacing the OLS linear-probability two-way partition.

Input : EVALUATION-807-video.zip  (uses panel_raw_judge_labels_full.csv)
Aggregation: paper rule -> 2 judges per report, BOTH must agree (AND),
             ties -> no hallucination.  Reproduces Table II.

Run:  pip install pandas numpy scipy statsmodels
      python limitation_v_vi.py /path/to/EVALUATION-807-video.zip
"""
import sys, zipfile, io
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2

# ZIP = sys.argv[1] if len(sys.argv) > 1 else \
ZIP = "/content/drive/Shareddrives/DR KOFI RESEARCH/EVALUATION-807-video.zip"
H = ["H1", "H2", "H3", "H4", "H5", "H6"]

# ----- technique grouping (edit here if you classify differently) -----
MULTI = {"True-Iterative", "Sequential", "Least-to-Most", "ReAct"}
SINGLE = {"Zero-Shot", "Chain-of-Thought", "Meta-Prompting", "Self-Consistency"}

# ---------------- load + aggregate (paper rule) ----------------
with zipfile.ZipFile(ZIP) as z:
    name = [n for n in z.namelist() if n.endswith("panel_raw_judge_labels_full.csv")][0]
    raw = pd.read_csv(io.BytesIO(z.read(name)))

# per report: a type is positive iff ALL judges (here 2) flag it; ties -> 0
rep = (raw.groupby(["model", "technique", "video"])[H]
          .agg(lambda s: int(s.sum() == len(s))).reset_index())
rep["any"] = rep[H].max(axis=1)
rep["count"] = rep[H].sum(axis=1)
rep["turn"] = np.where(rep.technique.isin(MULTI), "multi",
              np.where(rep.technique.isin(SINGLE), "single", "other"))

print(f"reports={len(rep)}  pooled ANY={rep['any'].mean()*100:.1f}%  "
      f"mean count={rep['count'].mean():.2f}  (should match Table II)\n")

# ============================================================
# (v) MULTI-TURN vs SINGLE-TURN  -- direct effect-size test
# ============================================================
print("=" * 60)
print("(v) Multi-turn vs single-turn hallucination")
print("=" * 60)
m = rep[rep.turn == "multi"]
s = rep[rep.turn == "single"]
print(f"multi-turn  : n={len(m):5d}  ANY={m['any'].mean()*100:5.1f}%  "
      f"mean count={m['count'].mean():.3f}  median={int(m['count'].median())}")
print(f"single-turn : n={len(s):5d}  ANY={s['any'].mean()*100:5.1f}%  "
      f"mean count={s['count'].mean():.3f}  median={int(s['count'].median())}")

# Mann-Whitney U on the per-report hallucination count
U, p = mannwhitneyu(m["count"], s["count"], alternative="two-sided")
# Cliff's delta from U (delta = 2U/(n1 n2) - 1)
delta = 2 * U / (len(m) * len(s)) - 1
mag = ("negligible" if abs(delta) < .147 else "small" if abs(delta) < .33
       else "medium" if abs(delta) < .474 else "large")
print(f"\nMann-Whitney U={U:.0f}  p={p:.3e}")
print(f"Cliff's delta={delta:+.3f}  ({mag}; +ve => multi-turn higher)")

# bootstrap 95% CI on the difference in ANY rate (multi - single)
rng = np.random.default_rng(0)
diffs = [rng.choice(m["any"].values, len(m)).mean()
         - rng.choice(s["any"].values, len(s)).mean() for _ in range(5000)]
lo, hi = np.percentile(diffs, [2.5, 97.5])
obs = m["any"].mean() - s["any"].mean()
print(f"ANY-rate diff (multi-single) = {obs*100:+.2f} pp  "
      f"95% CI [{lo*100:+.2f}, {hi*100:+.2f}] pp")

# per-axis (fabrication H1/H5, omission H3, distortion H2/H4/H6) quick view
axes = {"fabrication": ["H1", "H5"], "omission": ["H3"],
        "distortion": ["H2", "H4", "H6"]}
print("\nper-axis ANY rate (multi vs single):")
for ax, cols in axes.items():
    ma = m[cols].max(axis=1).mean() * 100
    sa = s[cols].max(axis=1).mean() * 100
    print(f"  {ax:11s}: multi {ma:5.1f}%  single {sa:5.1f}%  diff {ma-sa:+5.1f} pp")

# ============================================================
# (vi) LOGISTIC two-way model x technique on binary ANY
# ============================================================
print("\n" + "=" * 60)
print("(vi) Logistic regression: ANY ~ model * technique")
print("=" * 60)
import statsmodels.formula.api as smf

d = rep.copy()
def fit(formula):
    return smf.logit(formula, data=d).fit(disp=0)

null = fit("any ~ 1")
m_tech = fit("any ~ C(technique)")          # technique only
m_mod  = fit("any ~ C(model)")              # model only
m_add  = fit("any ~ C(model) + C(technique)")
m_full = fit("any ~ C(model) * C(technique)")

def lr(big, small, label):
    stat = 2 * (big.llf - small.llf)
    ddf = int(big.df_model - small.df_model)
    p = chi2.sf(stat, ddf)
    # McFadden contribution relative to null
    mcf = (big.llf - small.llf) / (0 - null.llf)
    print(f"  {label:22s} LR chi2={stat:9.1f}  df={ddf:2d}  "
          f"p={p:.2e}  pseudo-R2 add={mcf:.4f}")

print(f"  null log-lik = {null.llf:.1f}   full McFadden R2 = "
      f"{1 - m_full.llf/null.llf:.4f}\n")
lr(m_add,  m_tech, "model | technique")     # model main effect
lr(m_add,  m_mod,  "technique | model")     # technique main effect
lr(m_full, m_add,  "interaction")           # model x technique
print("\n(If model's LR chi2 >> technique's and the interaction is "
      "significant, the OLS ordering in RQ1 is confirmed under a logistic "
      "specification.)")

reports=19361  pooled ANY=91.1%  mean count=2.58  (should match Table II)

(v) Multi-turn vs single-turn hallucination
multi-turn  : n= 9684  ANY= 90.8%  mean count=2.658  median=3
single-turn : n= 9677  ANY= 91.4%  mean count=2.501  median=2

Mann-Whitney U=49613835  p=4.969e-13
Cliff's delta=+0.059  (negligible; +ve => multi-turn higher)
ANY-rate diff (multi-single) = -0.58 pp  95% CI [-1.36, +0.20] pp

per-axis ANY rate (multi vs single):
  fabrication: multi  72.9%  single  74.0%  diff  -1.2 pp
  omission   : multi  46.4%  single  40.9%  diff  +5.5 pp
  distortion : multi  61.1%  single  52.0%  diff  +9.1 pp

(vi) Logistic regression: ANY ~ model * technique
  null log-lik = -5807.6   full McFadden R2 = 0.1566

  model | technique      LR chi2=   1309.8  df= 2  p=3.73e-285  pseudo-R2 add=0.1128
  technique | model      LR chi2=    180.8  df= 7  p=1.34e-35  pseudo-R2 add=0.0156
  interaction            LR chi2=    338.2  df=14  p=1.23e-63  pseudo-R2 add=0.0291

(If model's LR chi2 >

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
